# 🏠 AhmedVall PropAI — Qatar Real Estate Multi-Agent System

> **Kaggle 5-Day AI Agents Intensive — Capstone Project**  
> **Track:** Agents for Business  
> **Author:** Ahmed Vall Jemal Dine Sidina

---

## 🎯 Problem Statement

Qatar's real estate market is one of the fastest-growing in the GCC region.  
Areas like **Lusail, Pearl Qatar, and West Bay** see quarterly price shifts of 3–5%,  
yet investors and brokers lack a unified tool that simultaneously:

1. **Estimates fair market value** based on property specs
2. **Analyzes market trends** for the target zone
3. **Generates a professional report** combining both insights

## 💡 Solution: Multi-Agent AI System

**AhmedVall PropAI** is a multi-agent system built with **Google ADK + Gemini**.

```
User Request → Orchestrator Agent
                    ↓
       ┌────────────┴────────────┐
       ↓                         ↓
 Valuation Agent      Market Research Agent
 (price_estimator,    (trend_analyzer,
  zone_lookup)         compare_zones)
       └────────────┬────────────┘
                    ↓
             Report Agent
                    ↓
       📊 Professional Valuation Report
```

| Concept | Implementation |
|---------|----------------|
| ✅ Multi-Agent System (ADK) | SequentialAgent + 3 specialized agents |
| ✅ MCP Server | Qatar zones data via MCP protocol pattern |
| ✅ Security Features | Kaggle Secrets + input validation |
| ✅ Agent Skills / Tools | 5 ADK FunctionTools |

In [14]:
# ── Step 1: Install Dependencies ─────────────────────────────────────────────
import subprocess, sys

pkgs = ['google-adk', 'nest_asyncio', 'rich']
for pkg in pkgs:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg}')

print('\n✅ All packages installed')

✅ google-adk
✅ nest_asyncio
✅ rich

✅ All packages installed


In [15]:
# ── Step 2: Imports & API Key (Secure) ───────────────────────────────────────
import os, json, asyncio
import nest_asyncio
from datetime import datetime
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.markdown import Markdown

nest_asyncio.apply()
console = Console()

# ✅ SECURE: Load from Kaggle Secrets — never hardcode API keys
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret('GEMINI_API_KEY')
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY
    console.print('[green]✅ API Key loaded securely from Kaggle Secrets[/green]')
except Exception as e:
    console.print(f'[yellow]⚠️  Secrets not available: {e}[/yellow]')
    # For local testing only — remove before sharing
    # os.environ['GEMINI_API_KEY'] = 'YOUR_KEY_HERE'

console.print(Panel(
    '[bold cyan]🏠 AhmedVall PropAI[/bold cyan]\n'
    '[white]Qatar Real Estate Multi-Agent Valuation System[/white]\n'
    '[dim]Google ADK 2.3.0 + Gemini 2.0 Flash[/dim]',
    border_style='cyan'
))

✅ API Key loaded securely from Kaggle Secrets

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🏠 AhmedVall PropAI                                                                                             │
│ Qatar Real Estate Multi-Agent Valuation System                                                                  │
│ Google ADK 2.3.0 + Gemini 2.0 Flash                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [16]:
# ── Step 3: MCP Server — Qatar Real Estate Data Layer ────────────────────────
"""
MCP Server implements Model Context Protocol pattern:
Data is served as a standalone service — agents never access data directly.
This separation makes the system modular, testable, and scalable.
"""

QATAR_ZONES_DB = {
    'west bay':            {'ar':'الخليج الغربي',       'municipality':'Doha',              'price_per_sqm':18000, 'tier':'luxury',      'demand':'very_high', 'trend':4.1, 'rental_yield':5.8},
    'musheireb':           {'ar':'المشيرب قلب الدوحة', 'municipality':'Doha',              'price_per_sqm':15500, 'tier':'luxury',      'demand':'high',      'trend':3.8, 'rental_yield':7.2},
    'pearl porto arabia':  {'ar':'اللؤلؤة بورتو أرابيا','municipality':'Doha',             'price_per_sqm':16500, 'tier':'luxury',      'demand':'very_high', 'trend':3.5, 'rental_yield':6.5},
    'pearl qanat quartier':{'ar':'اللؤلؤة قناة كارتيه','municipality':'Doha',              'price_per_sqm':16200, 'tier':'luxury',      'demand':'high',      'trend':3.2, 'rental_yield':6.8},
    'al sadd':             {'ar':'السد',                'municipality':'Doha',              'price_per_sqm':10500, 'tier':'mid_premium', 'demand':'high',      'trend':2.1, 'rental_yield':6.9},
    'bin mahmoud':         {'ar':'بن محمود',            'municipality':'Doha',              'price_per_sqm':10000, 'tier':'mid_premium', 'demand':'high',      'trend':1.9, 'rental_yield':7.0},
    'lusail marina':       {'ar':'لوسيل المارينا',      'municipality':'Al Daayen (Lusail)','price_per_sqm':14500, 'tier':'premium',     'demand':'very_high', 'trend':4.2, 'rental_yield':6.1},
    'lusail fox hills':    {'ar':'لوسيل الفوكس هيلز',  'municipality':'Al Daayen (Lusail)','price_per_sqm':13000, 'tier':'premium',     'demand':'high',      'trend':3.9, 'rental_yield':6.3},
    'lusail al yasmine':   {'ar':'لوسيل الياسمين',     'municipality':'Al Daayen (Lusail)','price_per_sqm':13200, 'tier':'premium',     'demand':'high',      'trend':4.0, 'rental_yield':6.0},
    'lusail al kharaiej':  {'ar':'لوسيل الخرائج',      'municipality':'Al Daayen (Lusail)','price_per_sqm':12800, 'tier':'premium',     'demand':'high',      'trend':3.7, 'rental_yield':6.2},
    'al waab':             {'ar':'الوعب',               'municipality':'Al Rayyan',          'price_per_sqm':11500, 'tier':'mid_premium', 'demand':'medium',    'trend':2.0, 'rental_yield':5.9},
    'khalifa city':        {'ar':'مدينة خليفة',         'municipality':'Al Rayyan',          'price_per_sqm':11800, 'tier':'mid_premium', 'demand':'medium',    'trend':1.8, 'rental_yield':5.7},
    'abu hamour':          {'ar':'أبو هامور',           'municipality':'Al Rayyan',          'price_per_sqm':10000, 'tier':'mid',         'demand':'medium',    'trend':1.5, 'rental_yield':6.0},
    'al wakra coastal':    {'ar':'الوكرة الساحلية',    'municipality':'Al Wakra',           'price_per_sqm': 9500, 'tier':'mid',         'demand':'medium',    'trend':2.3, 'rental_yield':5.5},
    'al wukair':           {'ar':'الوكير',              'municipality':'Al Wakra',           'price_per_sqm': 8800, 'tier':'mid',         'demand':'low',       'trend':1.2, 'rental_yield':5.2},
    'umm salal mohammed':  {'ar':'أم صلال محمد',       'municipality':'Umm Salal',          'price_per_sqm': 9000, 'tier':'mid',         'demand':'low',       'trend':0.9, 'rental_yield':5.0},
    'al khor':             {'ar':'الخور',               'municipality':'Al Khor',            'price_per_sqm': 8000, 'tier':'affordable',  'demand':'low',       'trend':0.7, 'rental_yield':4.8},
    'al ruwais':           {'ar':'الرويس',              'municipality':'Al Shamal',          'price_per_sqm': 7800, 'tier':'affordable',  'demand':'very_low',  'trend':0.5, 'rental_yield':4.5},
}

TYPE_MULTIPLIERS = {'apartment':1.00,'villa':1.15,'penthouse':1.35,'townhouse':1.08,'studio':0.90}
BEDROOM_FACTORS  = {1:0.92, 2:1.00, 3:1.06, 4:1.12, 5:1.18}


class QatarMCPServer:
    """MCP Server: Qatar Real Estate Data Provider"""

    def find_zone(self, query: str):
        q = query.lower().strip()
        if q in QATAR_ZONES_DB: return q, QATAR_ZONES_DB[q]
        for key, val in QATAR_ZONES_DB.items():
            if q in key or any(w in key.split() for w in q.split() if len(w)>3):
                return key, val
            if q in val['ar']: return key, val
        return None, None

    def get_zone_data(self, zone: str) -> dict:
        key, val = self.find_zone(zone)
        if not val:
            return {'ar':zone,'municipality':'Doha (est.)','price_per_sqm':11000,
                    'tier':'mid','demand':'medium','trend':2.0,'rental_yield':5.5,'estimated':True}
        return {**val, 'key':key}

    def estimate_price(self, zone, prop_type, size_sqm, bedrooms) -> dict:
        z = self.get_zone_data(zone)
        tm = TYPE_MULTIPLIERS.get(prop_type.lower(), 1.0)
        bf = BEDROOM_FACTORS.get(min(bedrooms, 5), 1.15)
        price = z['price_per_sqm'] * size_sqm * tm * bf
        return {
            'zone_ar': z['ar'], 'municipality': z['municipality'],
            'base_price_per_sqm': z['price_per_sqm'],
            'fair_value_qar': round(price, -3),
            'range_low_qar':  round(price*0.93, -3),
            'range_high_qar': round(price*1.07, -3),
            'tier': z['tier']
        }

    def analyze_trend(self, zone) -> dict:
        z = self.get_zone_data(zone)
        t = z['trend']
        signal = 'STRONG BUY' if t>=4.0 else 'BUY' if t>=2.5 else 'HOLD' if t>=1.5 else 'NEUTRAL'
        return {
            'zone_ar': z['ar'], 'municipality': z['municipality'],
            'quarterly_growth_pct': t, 'annual_projected_pct': round(t*4,1),
            'demand': z['demand'], 'rental_yield_pct': z['rental_yield'],
            'signal': signal
        }

    def compare_zones(self, zones: list) -> list:
        return sorted([
            {'zone': z, **{k:v for k,v in self.get_zone_data(z).items()
                          if k in ['ar','price_per_sqm','trend','demand','rental_yield','tier']}}
            for z in zones
        ], key=lambda x: x['price_per_sqm'], reverse=True)


mcp = QatarMCPServer()
console.print('[green]✅ MCP Server initialized[/green]')
console.print(f'   [dim]📊 {len(QATAR_ZONES_DB)} zones across 8 municipalities[/dim]')

✅ MCP Server initialized

📊 18 zones across 8 municipalities

In [17]:
# ── Step 4: ADK Tools (Agent Skills) ─────────────────────────────────────────
from google.adk.tools import FunctionTool

def validate_input(zone, prop_type, size_sqm, bedrooms):
    """Security: sanitize and validate all inputs before tool calls."""
    zone      = str(zone)[:100].strip()
    prop_type = str(prop_type)[:50].strip().lower()
    if prop_type not in TYPE_MULTIPLIERS: prop_type = 'apartment'
    size_sqm  = max(10.0, min(float(size_sqm), 10000.0))
    bedrooms  = max(1, min(int(bedrooms), 5))
    return zone, prop_type, size_sqm, bedrooms


def tool_price_estimator(zone: str, property_type: str, size_sqm: float, bedrooms: int) -> str:
    """Estimates fair market value of a Qatar property via MCP Server.
    Args:
        zone: Zone name (e.g. 'lusail marina', 'west bay', 'pearl porto arabia')
        property_type: 'apartment', 'villa', 'penthouse', 'townhouse', or 'studio'
        size_sqm: Area in square meters
        bedrooms: Number of bedrooms (1-5)
    Returns: JSON with fair value and price range in QAR"""
    z, pt, s, b = validate_input(zone, property_type, size_sqm, bedrooms)
    return json.dumps(mcp.estimate_price(z, pt, s, b), ensure_ascii=False)


def tool_zone_lookup(zone: str) -> str:
    """Retrieves complete data for a Qatar real estate zone from MCP Server.
    Args:
        zone: Zone name in English or Arabic
    Returns: JSON with zone price, demand, trend, rental yield"""
    return json.dumps(mcp.get_zone_data(str(zone)[:100]), ensure_ascii=False)


def tool_trend_analyzer(zone: str) -> str:
    """Analyzes market trend and investment signal for a Qatar zone.
    Args:
        zone: Zone name to analyze
    Returns: JSON with trend %, rental yield, and investment signal (STRONG BUY/BUY/HOLD/NEUTRAL)"""
    return json.dumps(mcp.analyze_trend(str(zone)[:100]), ensure_ascii=False)


def tool_compare_zones(zone1: str, zone2: str, zone3: str = '') -> str:
    """Compares 2-3 Qatar zones side by side on price, trend, yield.
    Args:
        zone1: First zone
        zone2: Second zone
        zone3: Optional third zone
    Returns: JSON with ranked zone comparison"""
    zones = [str(zone1)[:100], str(zone2)[:100]]
    if zone3: zones.append(str(zone3)[:100])
    return json.dumps(mcp.compare_zones(zones), ensure_ascii=False)


def tool_list_top_zones(count: int = 5) -> str:
    """Lists top Qatar real estate zones by price per sqm.
    Args:
        count: Number of top zones to return (default 5)
    Returns: JSON list of top zones with prices"""
    top = sorted(QATAR_ZONES_DB.items(), key=lambda x: x[1]['price_per_sqm'], reverse=True)
    return json.dumps([{'zone':k,'ar':v['ar'],'price_per_sqm':v['price_per_sqm'],
                        'trend':v['trend'],'rental_yield':v['rental_yield']}
                       for k,v in top[:int(count)]], ensure_ascii=False)


# Wrap as ADK FunctionTools
price_tool   = FunctionTool(func=tool_price_estimator)
lookup_tool  = FunctionTool(func=tool_zone_lookup)
trend_tool   = FunctionTool(func=tool_trend_analyzer)
compare_tool = FunctionTool(func=tool_compare_zones)
top_tool     = FunctionTool(func=tool_list_top_zones)

console.print('[green]✅ 5 ADK FunctionTools ready[/green]')
console.print('   [dim]price_estimator · zone_lookup · trend_analyzer · compare_zones · list_top_zones[/dim]')

✅ 5 ADK FunctionTools ready

price_estimator · zone_lookup · trend_analyzer · compare_zones · list_top_zones

In [18]:
# ── Step 5: Multi-Agent System (ADK) ─────────────────────────────────────────
from google.adk.agents import Agent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

MODEL = 'gemini-2.0-flash'

# Agent 1: Valuation
valuation_agent = Agent(
    name='valuation_agent',
    model=MODEL,
    description='Qatar property valuation specialist. Uses price_estimator and zone_lookup tools.',
    instruction=(
        'You are a Qatar real estate valuation expert. '
        'Always use zone_lookup first to get zone details, then price_estimator for fair value. '
        'Report exact numbers in QAR. Never invent prices — always use tool data.'
    ),
    tools=[price_tool, lookup_tool],
)

# Agent 2: Market Research
market_agent = Agent(
    name='market_research_agent',
    model=MODEL,
    description='Qatar market trend analyst. Uses trend_analyzer, compare_zones, list_top_zones.',
    instruction=(
        'You are a Qatar real estate market analyst. '
        'Use trend_analyzer for investment signals. Use compare_zones when asked to benchmark. '
        'Always include: quarterly growth %, rental yield %, and investment signal.'
    ),
    tools=[trend_tool, compare_tool, top_tool],
)

# Agent 3: Report Generator
report_agent = Agent(
    name='report_agent',
    model=MODEL,
    description='Synthesizes valuation and market data into professional property reports.',
    instruction=(
        'You are a professional real estate report writer. '
        'Combine the valuation and market analysis from previous agents into a structured report with: '
        '📋 Property Summary | 💰 Valuation | 📈 Market Analysis | 🏆 Investment Verdict | 💡 Recommendations. '
        f'End with: Report by AhmedVall PropAI · {datetime.now().strftime("%Y-%m-%d")}'
    ),
    tools=[],
)

# Orchestrator: SequentialAgent
orchestrator = SequentialAgent(
    name='propai_orchestrator',
    description='AhmedVall PropAI — coordinates valuation, market research, and report generation.',
    sub_agents=[valuation_agent, market_agent, report_agent],
)

# Session & Runner
session_service = InMemorySessionService()
runner = Runner(
    agent=orchestrator,
    app_name='AhmedVall_PropAI',
    session_service=session_service,
)

console.print('[green]✅ Multi-Agent System ready[/green]')
console.print('   [cyan]Orchestrator[/cyan] → [yellow]Valuation[/yellow] → [yellow]Market Research[/yellow] → [yellow]Report[/yellow]')

✅ Multi-Agent System ready

Orchestrator → Valuation → Market Research → Report

In [19]:
# ── Demo Mode: Full Pipeline with Real MCP Data ──────────────────────────────
"""
PropAI Demo: Uses real MCP Server data for all calculations.
The 3-agent pipeline (Valuation → Market Research → Report) is demonstrated
with structured outputs. When Gemini quota resets, swap simulate_propai()
with run_propai() for live LLM responses.
"""
from rich.markdown import Markdown
from datetime import datetime

def simulate_propai(zone: str, prop_type: str, size_sqm: float,
                    bedrooms: int, asking_price: int) -> str:
    """Full 3-agent pipeline simulation using real MCP Server."""

    # Agent 1: Valuation Agent
    console.print("\n[bold cyan]🏷️  Valuation Agent[/bold cyan] [dim]working...[/dim]")
    console.print("   [yellow]🔧 Tool:[/yellow] [green]zone_lookup[/green]")
    zone_data = mcp.get_zone_data(zone)
    console.print("   [yellow]🔧 Tool:[/yellow] [green]price_estimator[/green]")
    val = mcp.estimate_price(zone, prop_type, size_sqm, bedrooms)

    # Agent 2: Market Research Agent
    console.print("\n[bold cyan]📈 Market Research Agent[/bold cyan] [dim]working...[/dim]")
    console.print("   [yellow]🔧 Tool:[/yellow] [green]trend_analyzer[/green]")
    trend = mcp.analyze_trend(zone)

    # Deal assessment logic
    diff = ((asking_price - val["fair_value_qar"]) / val["fair_value_qar"]) * 100
    if diff < -7:   deal, deal_ar = "🟢 INVESTMENT OPPORTUNITY", "لقطة استثمارية"
    elif diff > 7:  deal, deal_ar = "🔴 OVERPRICED",              "سعر مرتفع"
    else:           deal, deal_ar = "🟡 FAIR PRICE",              "سعر عادل"

    if diff < -7:
        verdict = f"✅ **BUY** — Priced {abs(diff):.1f}% below fair value. Strong buy with {trend['rental_yield_pct']}% yield."
    elif diff > 7:
        verdict = f"❌ **NEGOTIATE** — Overpriced by {diff:.1f}%. Target {val['fair_value_qar']:,} QAR."
    else:
        verdict = f"⚠️ **HOLD** — Fair price range. Negotiate 5% down before committing."

    # Agent 3: Report Agent
    console.print("\n[bold cyan]📋 Report Agent[/bold cyan] [dim]synthesizing...[/dim]")

    return f"""
## 📋 Property Summary
| Field | Details |
|-------|---------|
| **Zone** | {val["zone_ar"]} |
| **Municipality** | {val["municipality"]} |
| **Type / Size** | {prop_type.title()} · {size_sqm:,.0f} m² · {bedrooms} BR |
| **Asking Price** | {asking_price:,} QAR |

## 💰 Valuation Results
| | QAR |
|--|-----|
| Range Low | {val["range_low_qar"]:,} |
| **Fair Value** | **{val["fair_value_qar"]:,}** |
| Range High | {val["range_high_qar"]:,} |
| Base Price/m² | {val["base_price_per_sqm"]:,} |

**Deal: {deal} ({deal_ar})** — Asking is {abs(diff):.1f}% {"above" if diff>0 else "below"} fair value.

## 📈 Market Analysis
| Indicator | Value |
|-----------|-------|
| Quarterly Growth | ▲ +{trend["quarterly_growth_pct"]}% |
| Annual Projection | ▲ +{trend["annual_projected_pct"]}% |
| Rental Yield | {trend["rental_yield_pct"]}% net |
| Demand | {trend["demand"].replace("_"," ").title()} |
| Signal | **{trend["signal"]}** |

## 🏆 Investment Verdict
{verdict}

## 💡 Recommendations
1. Target price: {val["fair_value_qar"]:,} QAR (fair value)
2. Zone projects +{trend["annual_projected_pct"]}% annual appreciation
3. Rental yield {trend["rental_yield_pct"]}% supports buy-to-let strategy
4. Verify: title deed, service charges, and finishing quality

---
*AhmedVall PropAI · Google ADK + MCP Server · {datetime.now().strftime("%Y-%m-%d")}*
"""

# ── Run all 3 Demos ───────────────────────────────────────────────────────────
console.rule("[cyan]🏠 Demo 1 — Villa in Lusail Al Yasmine[/cyan]")
console.print(Markdown(simulate_propai("lusail al yasmine", "villa", 450, 4, 5_200_000)))

console.rule("[cyan]🏠 Demo 2 — Apartment in West Bay + Comparison[/cyan]")
console.print(Markdown(simulate_propai("west bay", "apartment", 180, 3, 3_200_000)))

# Zone comparison for Demo 2
console.print("\n[bold yellow]Zone Comparison:[/bold yellow]")
comparison = mcp.compare_zones(["west bay", "pearl porto arabia", "musheireb"])
for z in comparison:
    console.print(f"  {z['ar']}: {z['price_per_sqm']:,} QAR/m² | Trend: +{z['trend']}% | Yield: {z['rental_yield']}%")

console.rule("[cyan]🏠 Demo 3 — Investment Strategy (2M QAR budget)[/cyan]")
top = mcp.compare_zones(["lusail marina", "musheireb", "al sadd", "al wakra coastal"])
best = min(top, key=lambda x: abs(x["price_per_sqm"] * 140 - 2_000_000))
console.print(f"\n[green]Best match for 2M budget: {best['ar']} ({best['price_per_sqm']:,} QAR/m²)[/green]")
console.print(Markdown(simulate_propai(best["zone"], "apartment", 140, 2, 1_900_000)))


───────────────────────────────────── 🏠 Demo 1 — Villa in Lusail Al Yasmine ──────────────────────────────────────

🏷️  Valuation Agent working...

🔧 Tool: zone_lookup

🔧 Tool: price_estimator

📈 Market Research Agent working...

...

🔧 Tool: price_estimator

📈 Market Research Agent working...

🔧 Tool: trend_analyzer

📋 Report Agent synthesizing...

📋 Property Summary                                                

                                            
  Field          Details                    
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Zone           لوسيل المارينا             
  Municipality   Al Daayen (Lusail)         
  Type / Size    Apartment · 140 m² · 2 BR  
  Asking Price   1,900,000 QAR              
                                            


                                               💰 Valuation Results                                                

                               
                  QAR          
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Range Low       1,888,000.0  
  Fair Value      2,030,000.0  
  Range High      2,172,000.0  
  Base Price/m²   14,500       
                               

Deal: 🟡 FAIR PRICE (سعر عادل) — Asking is 6.4% below fair value.                                                  


                                                📈 Market Analysis                                                 

                                  
  Indicator           Value       
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Quarterly Growth    ▲ +4.2%     
  Annual Projection   ▲ +16.8%    
  Rental Yield        6.1% net    
  Demand              Very High   
  Signal              STRONG BUY  
                                  


                                               🏆 Investment Verdict                                               

⚠️ HOLD — Fair price range. Negotiate 5% down before committing.                                                    


                                                💡 Recommendations                                                 

 1 Target price: 2,030,000.0 QAR (fair value)                                                                      
 2 Zone projects +16.8% annual appreciation                                                                        
 3 Rental yield 6.1% supports buy-to-let strategy                                                                  
 4 Verify: title deed, service charges, and finishing quality                                                      

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
AhmedVall PropAI · Google ADK + MCP Server · 2026-07-01

In [20]:
# ── Demo 3: Investment Strategy Query ────────────────────────────────────────
import os

# تنظيف المتغير المتضارب
os.environ.pop("GOOGLE_API_KEY", None)

# استدعاء الدالة الصحيحة المعرفة في الخطوة السابقة بمشروعك
report3 = simulate_propai("lusail marina", "apartment", 140, 2, 1_900_000)

console.print(Markdown(report3))

🏷️  Valuation Agent working...

🔧 Tool: zone_lookup

🔧 Tool: price_estimator

📈 Market Research Agent working...

🔧 Tool: trend_analyzer

📋 Report Agent synthesizing...

📋 Property Summary                                                

                                            
  Field          Details                    
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Zone           لوسيل المارينا             
  Municipality   Al Daayen (Lusail)         
  Type / Size    Apartment · 140 m² · 2 BR  
  Asking Price   1,900,000 QAR              
                                            


                                               💰 Valuation Results                                                

                               
                  QAR          
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Range Low       1,888,000.0  
  Fair Value      2,030,000.0  
  Range High      2,172,000.0  
  Base Price/m²   14,500       
                               

Deal: 🟡 FAIR PRICE (سعر عادل) — Asking is 6.4% below fair value.                                                  


                                                📈 Market Analysis                                                 

                                  
  Indicator           Value       
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  Quarterly Growth    ▲ +4.2%     
  Annual Projection   ▲ +16.8%    
  Rental Yield        6.1% net    
  Demand              Very High   
  Signal              STRONG BUY  
                                  


                                               🏆 Investment Verdict                                               

⚠️ HOLD — Fair price range. Negotiate 5% down before committing.                                                    


                                                💡 Recommendations                                                 

 1 Target price: 2,030,000.0 QAR (fair value)                                                                      
 2 Zone projects +16.8% annual appreciation                                                                        
 3 Rental yield 6.1% supports buy-to-let strategy                                                                  
 4 Verify: title deed, service charges, and finishing quality                                                      

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────
AhmedVall PropAI · Google ADK + MCP Server · 2026-07-01

In [21]:
# ── Step 7: Market Overview Dashboard ────────────────────────────────────────
console.rule('[bold cyan]📊 Qatar Real Estate Market Overview — 2026[/bold cyan]')

tbl = Table(title='AhmedVall PropAI — Zone Price Index',
            header_style='bold cyan', border_style='dim', show_lines=True)
tbl.add_column('Zone (AR)',       style='bold white', min_width=18)
tbl.add_column('Municipality',   style='dim',         min_width=14)
tbl.add_column('QAR/m²',         style='cyan',        justify='right', min_width=8)
tbl.add_column('Q-Trend',        style='green',       justify='center', min_width=9)
tbl.add_column('Yield',          style='yellow',      justify='center', min_width=7)
tbl.add_column('Signal',         style='bold',        justify='center', min_width=12)

for key, z in sorted(QATAR_ZONES_DB.items(), key=lambda x: x[1]['price_per_sqm'], reverse=True):
    t = z['trend']
    sig = ('[bold green]STRONG BUY[/bold green]' if t>=4.0 else
           '[green]BUY[/green]'        if t>=2.5 else
           '[yellow]HOLD[/yellow]'     if t>=1.5 else
           '[dim]NEUTRAL[/dim]')
    tbl.add_row(z['ar'], z['municipality'], f"{z['price_per_sqm']:,}",
                f'▲ +{t}%', f"{z['rental_yield']}%", sig)

console.print(tbl)
console.print(f'[dim]MCP Server data · AhmedVall PropAI · {datetime.now().strftime("%Y-%m-%d")}[/dim]')

─────────────────────────────────── 📊 Qatar Real Estate Market Overview — 2026 ───────────────────────────────────

                             AhmedVall PropAI — Zone Price Index                             
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Zone (AR)            ┃ Municipality       ┃   QAR/m² ┃  Q-Trend  ┃  Yield  ┃    Signal    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ الخليج الغربي        │ Doha               │   18,000 │  ▲ +4.1%  │  5.8%   │  STRONG BUY  │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ اللؤلؤة بورتو أرابيا │ Doha               │   16,500 │  ▲ +3.5%  │  6.5%   │     BUY      │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ اللؤلؤة قناة كارتيه  │ Doha               │   16,200 │  ▲ +3.2%  │  6.8%   │     BUY      │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ المشيرب قلب الدوحة   │ Doha               │   15,500 │  ▲ +3.8%  │  7.2%   │     BUY      │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ لوسيل المارينا       │ Al Daayen (Lusail) │   14,500 │  ▲ +4.2%  │  6.1%   │  STRONG BUY  │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ لوسيل الياسمين       │ Al Daayen (Lusail) │   13,200 │  ▲ +4.0%  │  6.0%   │  STRONG BUY  │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ لوسيل الفوكس هيلز    │ Al Daayen (Lusail) │   13,000 │  ▲ +3.9%  │  6.3%   │     BUY      │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ لوسيل الخرائج        │ Al Daayen (Lusail) │   12,800 │  ▲ +3.7%  │  6.2%   │     BUY      │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ مدينة خليفة          │ Al Rayyan          │   11,800 │  ▲ +1.8%  │  5.7%   │     HOLD     │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ الوعب                │ Al Rayyan          │   11,500 │  ▲ +2.0%  │  5.9%   │     HOLD     │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ السد                 │ Doha               │   10,500 │  ▲ +2.1%  │  6.9%   │     HOLD     │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ بن محمود             │ Doha               │   10,000 │  ▲ +1.9%  │  7.0%   │     HOLD     │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ أبو هامور            │ Al Rayyan          │   10,000 │  ▲ +1.5%  │  6.0%   │     HOLD     │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ الوكرة الساحلية      │ Al Wakra           │    9,500 │  ▲ +2.3%  │  5.5%   │     HOLD     │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ أم صلال محمد         │ Umm Salal          │    9,000 │  ▲ +0.9%  │  5.0%   │   NEUTRAL    │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ الوكير               │ Al Wakra           │    8,800 │  ▲ +1.2%  │  5.2%   │   NEUTRAL    │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ الخور                │ Al Khor            │    8,000 │  ▲ +0.7%  │  4.8%   │   NEUTRAL    │
├──────────────────────┼────────────────────┼──────────┼───────────┼─────────┼──────────────┤
│ الرويس               │ Al Shamal          │    7,800 │  ▲ +0.5%  │  4.5%   │   NEUTRAL    │
└──────────────────────┴────────────────────┴──────────┴───────────┴─────────┴──────────────┘

MCP Server data · AhmedVall PropAI · 2026-07-01

In [22]:
# ── Step 8: Security Features Demo ───────────────────────────────────────────
console.rule('[bold red]🔒 Security Architecture[/bold red]')

features = [
    ('API Key via Kaggle Secrets',  'UserSecretsClient() — never hardcoded in source'),
    ('Input Validation',            'validate_input() sanitizes all tool inputs before MCP calls'),
    ('String Length Limits',        'All strings capped at 100/50 chars to prevent injection'),
    ('Numeric Range Checks',        'size: 10-10000m², bedrooms: 1-5, enforced in validate_input()'),
    ('Safe Type Defaults',          "Invalid property_type → 'apartment' (never raises exception)"),
    ('Data/Logic Separation',       'MCP Server isolates data from agent logic completely'),
    ('Session Isolation',           'InMemorySessionService — each user session is independent'),
    ('Zero Credentials in Code',    'No API keys, passwords, or secrets in any code cell'),
]

sec = Table(header_style='bold red', border_style='dim', show_lines=True)
sec.add_column('Feature',        style='bold white', min_width=24)
sec.add_column('Implementation', style='dim',        min_width=52)
sec.add_column('✓',              justify='center',   min_width=4)
for f, i in features:
    sec.add_row(f, i, '[green]✅[/green]')
console.print(sec)

# Live test
console.print('\n[bold yellow]Live Security Test:[/bold yellow]')
tests = [
    ('lusail', 'apartment', -50,  3),
    ('west bay', 'mansion', 200,  15),
    ('pearl',   'villa',    300,   3),
]

──────────────────────────────────────────── 🔒 Security Architecture ─────────────────────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Feature                    ┃ Implementation                                                ┃  ✓   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ API Key via Kaggle Secrets │ UserSecretsClient() — never hardcoded in source               │  ✅  │
├────────────────────────────┼───────────────────────────────────────────────────────────────┼──────┤
│ Input Validation           │ validate_input() sanitizes all tool inputs before MCP calls   │  ✅  │
├────────────────────────────┼───────────────────────────────────────────────────────────────┼──────┤
│ String Length Limits       │ All strings capped at 100/50 chars to prevent injection       │  ✅  │
├────────────────────────────┼───────────────────────────────────────────────────────────────┼──────┤
│ Numeric Range Checks       │ size: 10-10000m², bedrooms: 1-5, enforced in validate_input() │  ✅  │
├────────────────────────────┼───────────────────────────────────────────────────────────────┼──────┤
│ Safe Type Defaults         │ Invalid property_type → 'apartment' (never raises exception)  │  ✅  │
├────────────────────────────┼───────────────────────────────────────────────────────────────┼──────┤
│ Data/Logic Separation      │ MCP Server isolates data from agent logic completely          │  ✅  │
├────────────────────────────┼───────────────────────────────────────────────────────────────┼──────┤
│ Session Isolation          │ InMemorySessionService — each user session is independent     │  ✅  │
├────────────────────────────┼───────────────────────────────────────────────────────────────┼──────┤
│ Zero Credentials in Code   │ No API keys, passwords, or secrets in any code cell           │  ✅  │
└────────────────────────────┴───────────────────────────────────────────────────────────────┴──────┘

Live Security Test: